# STRAT-412: Whole Foods Market Store Directory Scraper

This notebook scrapes the entire Whole Foods Market store directory and exports the data to a CSV file.

**Note:** Whole Foods uses a JavaScript-rendered store locator, so we try cloudscraper first. If that fails to find store links, we fall back to Selenium with headless Chrome.

**Output CSV columns:** Store Name, Store Number, Store Complex, Address, City, State, Zip, Phone Number

## Imports & Setup
Install required packages and configure cloudscraper. Also install Selenium/ChromeDriver as a fallback for dynamic pages.

In [ ]:
# Install required packages
!pip install cloudscraper beautifulsoup4 lxml selenium
!apt-get update -qq
!apt-get install -y -qq chromium-chromedriver

import cloudscraper
from bs4 import BeautifulSoup
import csv
import json
import time
import re

# Base URL for the Whole Foods website
BASE_URL = "https://www.wholefoodsmarket.com"

# Create a cloudscraper session
scraper = cloudscraper.create_scraper(
    browser={
        "browser": "chrome",
        "platform": "windows",
        "desktop": True,
    }
)


def get_soup(url, wait_seconds=1.5):
    """
    Fetches a URL using cloudscraper and returns a BeautifulSoup object.
    """
    time.sleep(wait_seconds)
    response = scraper.get(url)
    response.raise_for_status()
    return BeautifulSoup(response.text, "lxml")


# --- Selenium fallback setup (for JS-rendered pages) ---
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service

chrome_options = Options()
chrome_options.add_argument("--headless=new")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--window-size=1920,1080")
chrome_options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
)

driver = None  # initialized only if needed


def get_soup_selenium(url, wait_seconds=5):
    """
    Fallback: uses Selenium to load a JS-rendered page.
    """
    global driver
    if driver is None:
        driver = webdriver.Chrome(options=chrome_options)
    driver.get(url)
    time.sleep(wait_seconds)
    return BeautifulSoup(driver.page_source, "lxml")


# Quick test
test_soup = get_soup(BASE_URL + "/stores")
test_title = test_soup.find("title")
print("Page title: " + (test_title.get_text(strip=True) if test_title else "No title found"))
print("Page length: " + str(len(test_soup.get_text())) + " characters")
print("Setup complete!")

## Discovery: Inspect Raw HTML
Run this cell to see what the Whole Foods store page looks like. We check both cloudscraper and Selenium to see which gives us usable content.

In [ ]:
# Try cloudscraper first on a known store page
discovery_url = BASE_URL + "/stores/arlington"
print("=== TRYING CLOUDSCRAPER ===")
try:
    discovery_soup = get_soup(discovery_url, wait_seconds=3)
    print("Page length: " + str(len(discovery_soup.get_text())) + " chars")
    print(discovery_soup.prettify()[:3000])
except Exception as e:
    print("Cloudscraper failed: " + str(e))
    discovery_soup = None

# Check for JSON-LD structured data
print("\n=== JSON-LD STRUCTURED DATA ===")
if discovery_soup:
    json_ld_scripts = discovery_soup.find_all("script", type="application/ld+json")
    for i, script in enumerate(json_ld_scripts):
        try:
            data = json.loads(script.string)
            print("\nJSON-LD block " + str(i + 1) + ":")
            print(json.dumps(data, indent=2)[:2000])
        except Exception:
            print("Could not parse JSON-LD block " + str(i + 1))

# If cloudscraper returned very little content, try Selenium
if not discovery_soup or len(discovery_soup.get_text()) < 500:
    print("\n=== TRYING SELENIUM FALLBACK ===")
    try:
        sel_soup = get_soup_selenium(discovery_url)
        print("Selenium page length: " + str(len(sel_soup.get_text())) + " chars")
        print(sel_soup.prettify()[:3000])
    except Exception as e:
        print("Selenium also failed: " + str(e))

# Check for store links on the main /stores page
print("\n=== STORE LINKS ON /stores ===")
stores_soup = get_soup(BASE_URL + "/stores", wait_seconds=3)
store_links = []
for link in stores_soup.find_all("a", href=True):
    href = link["href"]
    if "/stores/" in href and href != "/stores/" and href != "/stores":
        store_links.append(href)
print("Found " + str(len(store_links)) + " store-related links")
for sl in store_links[:20]:
    print("  " + sl)

## Code Block #1: Scrape Location Info for One Store
This function extracts all 8 required fields from a single Whole Foods store page.

**Extraction strategy:**
1. JSON-LD structured data - most reliable
2. Schema.org microdata (itemprop attributes) - fallback
3. HTML elements (h1, address tags) - secondary fallback
4. Regex patterns - last resort

In [ ]:
def scrape_one_store(url, use_selenium=False):
    """
    Scrapes a single Whole Foods store page and returns a dictionary with:
    Store Name, Store Number, Store Complex, Address, City, State, Zip, Phone Number
    """
    if use_selenium:
        soup = get_soup_selenium(url)
    else:
        soup = get_soup(url)

    # Initialize all fields with empty defaults
    store_name = ""
    store_number = ""
    store_complex = ""
    address = ""
    city = ""
    state = ""
    zipcode = ""
    phone = ""

    # --- Try JSON-LD structured data first ---
    json_ld_scripts = soup.find_all("script", type="application/ld+json")
    for script in json_ld_scripts:
        try:
            data = json.loads(script.string)
            if isinstance(data, list):
                data = data[0]
            if "address" in data:
                addr_data = data["address"]
                address = addr_data.get("streetAddress", "")
                city = addr_data.get("addressLocality", "")
                state = addr_data.get("addressRegion", "")
                zipcode = addr_data.get("postalCode", "")
            if "telephone" in data:
                phone = data["telephone"]
            if "name" in data:
                store_name = data["name"]
            if "branchCode" in data:
                store_number = data["branchCode"]
        except (json.JSONDecodeError, TypeError, KeyError):
            continue

    # --- Store Name fallback ---
    if not store_name:
        h1 = soup.find("h1")
        if h1:
            store_name = h1.get_text(strip=True)

    # --- Store Number fallback ---
    if not store_number:
        page_text = soup.get_text()
        store_num_match = re.search(r"Store\s*#?\s*(\d+)", page_text)
        if store_num_match:
            store_number = store_num_match.group(1)

    # --- Address fallback: itemprop ---
    if not address:
        street_el = soup.find(attrs={"itemprop": "streetAddress"})
        if street_el:
            address = street_el.get_text(strip=True)
    if not city:
        city_el = soup.find(attrs={"itemprop": "addressLocality"})
        if city_el:
            city = city_el.get_text(strip=True)
    if not state:
        state_el = soup.find(attrs={"itemprop": "addressRegion"})
        if state_el:
            state = state_el.get_text(strip=True)
    if not zipcode:
        zip_el = soup.find(attrs={"itemprop": "postalCode"})
        if zip_el:
            zipcode = zip_el.get_text(strip=True)

    # --- Address fallback: address tag ---
    if not address:
        addr_tag = soup.find("address")
        if addr_tag:
            lines = [line.strip() for line in addr_tag.get_text().split("\n") if line.strip()]
            if lines:
                address = lines[0]
            if len(lines) >= 2:
                csz_match = re.match(r"(.+?),\s*([A-Z]{2})\s*(\d{5})", lines[1])
                if csz_match and not city:
                    city = csz_match.group(1)
                    state = csz_match.group(2)
                    zipcode = csz_match.group(3)

    # --- Phone fallback ---
    if not phone:
        phone_el = soup.find(attrs={"itemprop": "telephone"})
        if phone_el:
            phone = phone_el.get_text(strip=True)
    if not phone:
        tel_link = soup.find("a", href=re.compile(r"^tel:"))
        if tel_link:
            phone = tel_link.get_text(strip=True)
            if not phone:
                phone = tel_link["href"].replace("tel:", "").strip()
    if not phone:
        page_text = soup.get_text()
        phone_match = re.search(r"\(?\d{3}\)?[\s.-]?\d{3}[\s.-]?\d{4}", page_text)
        if phone_match:
            phone = phone_match.group(0)

    # --- Store Complex ---
    complex_el = (
        soup.find(class_=re.compile(r"complex|plaza|center|shopping", re.I))
        or soup.find(class_=re.compile(r"subtitle|subheading|sub-title", re.I))
    )
    if complex_el:
        candidate = complex_el.get_text(strip=True)
        if candidate and candidate != store_name:
            store_complex = candidate
    if not store_complex:
        h2_tags = soup.find_all("h2")
        for h2 in h2_tags:
            text = h2.get_text(strip=True)
            if re.search(r"Plaza|Center|Village|Square|Mall|Shopping|Market|Commons", text, re.I):
                store_complex = text
                break

    # --- Data Cleaning ---
    zip_match = re.search(r"\d{5}", zipcode)
    if zip_match:
        zipcode = zip_match.group(0)
    state = state.strip().upper()[:2]
    phone = phone.strip()

    return {
        "Store Name": store_name,
        "Store Number": store_number,
        "Store Complex": store_complex,
        "Address": address,
        "City": city,
        "State": state,
        "Zip": zipcode,
        "Phone Number": phone,
    }


# Test with one store
print("=" * 60)
print("SCRAPING ONE STORE")
print("=" * 60)
one_store_url = BASE_URL + "/stores/arlington"
result = scrape_one_store(one_store_url)
print("Using cloudscraper:")
for key, value in result.items():
    print("  " + key + ": " + value)

# If cloudscraper returned mostly empty, try Selenium
if not result["Address"]:
    print("\nCloudscraper returned no address. Trying Selenium...")
    result = scrape_one_store(one_store_url, use_selenium=True)
    USE_SELENIUM = True
    for key, value in result.items():
        print("  " + key + ": " + value)
else:
    USE_SELENIUM = False

print("\nUsing Selenium for remaining scrapes: " + str(USE_SELENIUM))

## Code Block #2: Test with a Small Loop of 3-4 URLs
Testing the scrape function on a few known store URLs from different states.

In [ ]:
# Define 4 test store URLs from different locations
test_urls = [
    BASE_URL + "/stores/arlington",
    BASE_URL + "/stores/austin-domain",
    BASE_URL + "/stores/brooklyn-third-and-third",
    BASE_URL + "/stores/chicago-lakeview",
]

print("=" * 60)
print("TESTING WITH SMALL LOOP (4 URLs)")
print("=" * 60)

test_results = []
for url in test_urls:
    print("\nScraping: " + url)
    try:
        store_data = scrape_one_store(url, use_selenium=USE_SELENIUM)
        test_results.append(store_data)
        for key, value in store_data.items():
            print("  " + key + ": " + value)
    except Exception as e:
        print("  ERROR: " + str(e))

print("\nSuccessfully scraped " + str(len(test_results)) + " out of " + str(len(test_urls)) + " test stores.")

## Code Block #3: Print the List of State URLs
Whole Foods does not use a state-based directory like Sprouts or Trader Joe's. Instead, the store locator at `/stores` lists individual store pages directly.

We attempt to find all store page links from the main `/stores` page, the sitemap, and a hardcoded list of known store slugs as a fallback.

In [ ]:
print("=" * 60)
print("GETTING STATE/REGION URLs")
print("=" * 60)

# Whole Foods does not have state-level directory pages like Sprouts.
# Their store locator is at /stores and lists stores directly.
# We will collect store URLs from multiple sources.

# Approach 1: Check if the sitemap has store URLs
print("Checking sitemap for store URLs...")
sitemap_store_urls = []
for sitemap_path in ["/sitemap.xml", "/sitemap_index.xml", "/sitemap-stores.xml"]:
    try:
        sitemap_soup = get_soup(BASE_URL + sitemap_path, wait_seconds=2)
        for loc in sitemap_soup.find_all("loc"):
            loc_text = loc.get_text(strip=True)
            # Match store URLs like /stores/store-slug
            if "/stores/" in loc_text and loc_text != BASE_URL + "/stores" and loc_text != BASE_URL + "/stores/":
                # Skip non-store pages like /stores/new-store-openings
                slug = loc_text.split("/stores/")[-1].rstrip("/")
                if slug and not any(x in slug for x in ["new-store", "site-map", "search", "find"]):
                    if loc_text not in sitemap_store_urls:
                        sitemap_store_urls.append(loc_text)
        if sitemap_store_urls:
            print("  Found " + str(len(sitemap_store_urls)) + " store URLs from " + sitemap_path)
            break
    except Exception as e:
        print("  Could not fetch " + sitemap_path + ": " + str(e))

# Approach 2: Scrape the /stores page for links
print("\nScraping /stores page for links...")
page_store_urls = []
try:
    # Try cloudscraper first
    stores_soup = get_soup(BASE_URL + "/stores", wait_seconds=3)
    for link in stores_soup.find_all("a", href=True):
        href = link["href"]
        if re.match(r"^/stores/[a-z][\w-]+$", href):
            slug = href.split("/stores/")[-1]
            if slug and not any(x in slug for x in ["new-store", "site-map", "search", "find"]):
                full_url = BASE_URL + href
                if full_url not in page_store_urls:
                    page_store_urls.append(full_url)
    print("  Found " + str(len(page_store_urls)) + " store URLs from /stores page (cloudscraper)")
except Exception as e:
    print("  Cloudscraper failed on /stores: " + str(e))

# If cloudscraper found very few, try Selenium
if len(page_store_urls) < 10:
    print("  Trying Selenium on /stores page...")
    try:
        sel_soup = get_soup_selenium(BASE_URL + "/stores", wait_seconds=8)
        for link in sel_soup.find_all("a", href=True):
            href = link["href"]
            if re.match(r"^/stores/[a-z][\w-]+$", href):
                slug = href.split("/stores/")[-1]
                if slug and not any(x in slug for x in ["new-store", "site-map", "search", "find"]):
                    full_url = BASE_URL + href
                    if full_url not in page_store_urls:
                        page_store_urls.append(full_url)
        print("  Found " + str(len(page_store_urls)) + " store URLs total after Selenium")
    except Exception as e:
        print("  Selenium also failed: " + str(e))

# Combine all sources
all_found_urls = list(set(sitemap_store_urls + page_store_urls))
all_found_urls.sort()

print("\nTotal unique store URLs found: " + str(len(all_found_urls)))

## Code Block #4: Print the List of City URLs
Since Whole Foods does not use a state/city hierarchy, this cell shows all the store page URLs grouped by the state extracted from each store's data. If we found URLs above, we display them here. If not, we note that the store locator requires dynamic interaction.

In [ ]:
print("=" * 60)
print("STORE URLs (Whole Foods does not use city-level directory pages)")
print("=" * 60)

# Whole Foods uses flat /stores/{slug} URLs rather than /stores/{state}/{city}/
# So city_urls is effectively the same as store_urls for this scraper.
city_urls = all_found_urls  # for consistency with assignment structure

print("Total store page URLs: " + str(len(city_urls)))
print("")
for url in city_urls:
    print("  " + url)

## Code Block #5: Print the List of Store URLs
Final deduplicated list of all individual Whole Foods store page URLs to scrape.

In [ ]:
print("=" * 60)
print("FINAL STORE URLs LIST")
print("=" * 60)

store_urls = all_found_urls
store_urls.sort()

print("Found " + str(len(store_urls)) + " store URLs to scrape:\n")
for url in store_urls:
    print("  " + url)

if len(store_urls) == 0:
    print("\nWARNING: No store URLs found automatically.")
    print("The Whole Foods store locator requires JavaScript interaction.")
    print("You may need to manually collect store URLs or use an alternative data source.")

## Code Block #6: Loop Through All Store URLs
Scrape every individual store page and collect the data. Progress is printed every 25 stores.

**Expected:** ~529 Whole Foods stores across the US.

In [ ]:
print("=" * 60)
print("SCRAPING ALL STORES")
print("=" * 60)

all_stores = []
errors = []

total = len(store_urls)
print("Total store URLs to scrape: " + str(total))

if total == 0:
    print("No store URLs to scrape. Check the discovery and URL collection cells above.")
else:
    for i, url in enumerate(store_urls, start=1):
        if i % 25 == 0 or i == 1:
            print("  Progress: " + str(i) + "/" + str(total) + " stores scraped...")
        try:
            store_data = scrape_one_store(url, use_selenium=USE_SELENIUM)
            store_data["URL"] = url
            all_stores.append(store_data)
        except Exception as e:
            print("  ERROR on " + url + ": " + str(e))
            errors.append({"url": url, "error": str(e)})

    print("\nDone! Successfully scraped " + str(len(all_stores)) + " stores.")
    if errors:
        print("Encountered " + str(len(errors)) + " errors:")
        for err in errors:
            print("  " + err["url"] + ": " + err["error"])

## Code Block #7: Write to CSV and Export
Write the collected store data to a CSV file and download it.

In [ ]:
csv_filename = "whole_foods_stores.csv"
csv_columns = [
    "Store Name",
    "Store Number",
    "Store Complex",
    "Address",
    "City",
    "State",
    "Zip",
    "Phone Number",
]

with open(csv_filename, "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=csv_columns, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(all_stores)

print("CSV file '" + csv_filename + "' written with " + str(len(all_stores)) + " rows.")
print("Columns: " + ", ".join(csv_columns))

print("\nPreview (first 5 rows):")
for store in all_stores[:5]:
    print("  " + store["Store Name"] + " | #" + store["Store Number"] + " | "
          + store["Address"] + ", " + store["City"] + ", " + store["State"] + " " + store["Zip"]
          + " | " + store["Phone Number"])

# Clean up Selenium driver if it was used
if driver is not None:
    driver.quit()
    print("\nSelenium driver closed.")

# Download the CSV in Google Colab
from google.colab import files
files.download(csv_filename)